# Green Metrics – PUE / WUE simulation (notebook version)

Computes hourly **Power Usage Effectiveness (PUE)** and **Water Usage Effectiveness (WUE)** of a data center
for any weather file, using Latin Hypercube sampled design parameters.

**How to use with your own data** – only edit the *Settings* cell:
1. Put your weather file(s) (`.epw`, or `.csv` with columns `T_oa` °C, `RH_oa` %, `P_oa` Pa) in a folder.
2. Choose the cooling model (`SIM_FUNC`) and a bounds/run CSV pair (copy one from `config/lhs/<region>/`).
3. Run all cells.

Run this notebook with the project's `.venv` kernel (see `docs/SETUP_GUIDE.md`, step 7).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import green_metrics.simulation.dc as dc
from green_metrics.batch.registry import SCENARIOS
from green_metrics.batch.run import read_weather, simulate_sample
from green_metrics.sampling.generate import generate_lhs_rows
from green_metrics.sampling.lhs_spec import load_lhs_spec

print("Built-in scenarios:", ", ".join(sorted(SCENARIOS)))

## 1. Settings (edit this cell)

In [ ]:
PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# --- choose a built-in scenario as the starting point ... -----------------------------
SCENARIO = "usa_dx"                 # e.g. sri_lanka_chiller, usa_air_chiller (see list above)

# --- ... and override anything for your own data ---------------------------------------
WEATHER_DIR = PROJECT / "data/epw_files/usa"         # folder of .epw / .csv files
BOUNDS_CSV  = PROJECT / "config/lhs/usa/dx_bounds.csv"
RUN_CSV     = PROJECT / "config/lhs/usa/dx_run.csv"  # n_samples, random_seed
MODEL_DIR   = PROJECT / "data/pkl_files"             # COP_2.pkl, COP_DX.pkl, COP_AC.pkl
OUTPUT_DIR  = PROJECT / "output/notebook"

N_SAMPLES   = 5       # None = use n_samples from RUN_CSV
MAX_HOURS   = 168     # quick demo = first week. Set to None for the full year (slow with many samples)
SAVE_EXCEL  = True

## 2. Latin Hypercube samples

In [ ]:
cfg = SCENARIOS[SCENARIO]
spec = load_lhs_spec(BOUNDS_CSV, RUN_CSV)
if N_SAMPLES:
    spec.n_samples = N_SAMPLES
samples = pd.DataFrame(generate_lhs_rows(spec))
names = pd.read_csv(BOUNDS_CSV, encoding="utf-8-sig")["name"].tolist()
samples.columns = names
print(f"{len(samples)} samples, {len(spec.active_positions)} varying parameters")
samples.head()

## 3. Load weather

In [ ]:
weather_files = sorted(p for p in WEATHER_DIR.iterdir() if p.suffix.lower() in (".epw", ".csv"))
print("Weather files:", [p.name for p in weather_files])
weather = {p.stem: read_weather(p).head(MAX_HOURS) for p in weather_files}
next(iter(weather.values())).describe().T

## 4. Simulate

In [ ]:
dc.configure_model_directory(MODEL_DIR)
fn = getattr(dc, cfg.sim_func)

results = {}   # (location, sample number) -> DataFrame
for loc, wdf in weather.items():
    for i, sample in samples.iterrows():
        results[(loc, i + 1)] = simulate_sample(
            fn, wdf, sample, cfg.pue_column, cfg.wue_column, desc=f"{loc} #{i + 1}")
print("done:", len(results), "runs")

## 5. Summary

In [ ]:
rows = [
    dict(location=loc, sample=n, PUE_mean=df[cfg.pue_column].mean(), PUE_max=df[cfg.pue_column].max(),
         WUE_mean=df[cfg.wue_column].mean(), WUE_max=df[cfg.wue_column].max())
    for (loc, n), df in results.items()
]
summary = pd.DataFrame(rows)
summary.groupby("location")[["PUE_mean", "WUE_mean"]].agg(["mean", "min", "max"]).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for (loc, n), df in results.items():
    axes[0].plot(df["Hour"], df[cfg.pue_column], alpha=.4)
    axes[1].plot(df["Hour"], df[cfg.wue_column], alpha=.4)
axes[0].set(title="PUE by hour (all locations/samples)", xlabel="hour", ylabel="PUE")
axes[1].set(title="WUE by hour (L/kWh)", xlabel="hour", ylabel="WUE")
plt.tight_layout(); plt.show()

## 6. Save to Excel (same layout as the command line tool)

In [ ]:
if SAVE_EXCEL:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for loc in weather:
        path = OUTPUT_DIR / f"{loc}.xlsx"
        with pd.ExcelWriter(path, engine="openpyxl") as writer:
            for (l, n), df in results.items():
                if l == loc:
                    df.to_excel(writer, sheet_name=f"{cfg.sheet_prefix}_{n}", index=False)
        print("saved", path)